In [ ]:
import torch
import torchvision
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
import math
from pathlib import Path
from PIL import Image
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import random
import numpy as np
import timm
from tqdm.auto import tqdm
import copy
import gc
from torch.amp import autocast, GradScaler

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

In [ ]:
class NICOPPDataset(Dataset):
    def __init__(
        self,
        root_path,
        domain_groups,
        splits=("train",),
        real_domains=None,
        class_to_idx=None,
        transform=None
    ):
        self.root_path = Path(root_path)
        self.domain_groups = domain_groups if isinstance(domain_groups, (list, tuple)) else [domain_groups]
        self.splits = splits if isinstance(splits, (list, tuple)) else [splits]
        self.real_domains = set(real_domains) if real_domains is not None else None
        self.transform = transform
        self.samples = []

        # Build class mapping
        if class_to_idx is None:
            class_names = set()

            for domain_group in self.domain_groups:
                for split in self.splits:
                    split_path = self.root_path / domain_group / split
                    if not split_path.exists():
                        continue

                    for real_domain_dir in split_path.iterdir():
                        if not real_domain_dir.is_dir():
                            continue

                        if self.real_domains is not None and real_domain_dir.name not in self.real_domains:
                            continue

                        for class_dir in real_domain_dir.iterdir():
                            if class_dir.is_dir():
                                class_names.add(class_dir.name)

            self.class_to_idx = {
                class_name: idx
                for idx, class_name in enumerate(sorted(class_names))
            }
        else:
            self.class_to_idx = class_to_idx

        # Collect samples
        for domain_group in self.domain_groups:
            for split in self.splits:
                split_path = self.root_path / domain_group / split
                if not split_path.exists():
                    continue

                for real_domain_dir in split_path.iterdir():
                    if not real_domain_dir.is_dir():
                        continue

                    real_domain = real_domain_dir.name

                    if self.real_domains is not None and real_domain not in self.real_domains:
                        continue

                    for class_dir in real_domain_dir.iterdir():
                        if not class_dir.is_dir():
                            continue

                        class_name = class_dir.name

                        if class_name not in self.class_to_idx:
                            continue

                        label = self.class_to_idx[class_name]

                        for img_path in class_dir.rglob("*"):
                            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                                self.samples.append({
                                    "image_path": img_path,
                                    "label": label,
                                    "class_name": class_name,
                                    "domain_group": domain_group,
                                    "real_domain": real_domain,
                                    "split": split,
                                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = sample["label"]
        return image, label

In [ ]:
class TimmFeaturizer(nn.Module):
    """
    Generic timm feature extractor for DomainBed-style algorithms.

    Important fix:
    Some timm models report backbone.num_features != actual output dim
    after num_classes=0/global_pool="avg". MobileNetV4 can report 960,
    while the real forward output is 1280. So we infer n_outputs by doing
    a real dummy forward pass instead of trusting .num_features.
    """
    def __init__(
        self,
        model_path: str,
        pretrained: bool = False,
        input_size: int = 224,
    ):
        super().__init__()

        if model_path is None:
            raise ValueError("model_path must be specified.")

        self.backbone = timm.create_model(
            model_path,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        # Robustly infer the actual feature dimension produced by forward().
        self.n_outputs = self._infer_n_outputs(input_size=input_size)

    def _flatten_features(self, features):
        if isinstance(features, (tuple, list)):
            features = features[-1]

        if features.ndim > 2:
            features = torch.flatten(
                torch.nn.functional.adaptive_avg_pool2d(features, 1),
                1,
            )

        return features

    def _infer_n_outputs(self, input_size: int = 224):
        was_training = self.backbone.training
        self.backbone.eval()

        with torch.no_grad():
            dummy = torch.zeros(1, 3, input_size, input_size)
            features = self.backbone(dummy)
            features = self._flatten_features(features)

        if was_training:
            self.backbone.train()

        if features.ndim != 2:
            raise ValueError(
                f"Expected 2D feature output [B, D], got shape {tuple(features.shape)}"
            )

        return features.shape[1]

    def forward(self, x):
        features = self.backbone(x)
        features = self._flatten_features(features)
        return features


class SagNetModel(nn.Module):
    """
    DomainBed-style SagNet using a generic timm backbone.

    - network_f: shared featurizer
    - network_c: content classifier
    - network_s: style classifier
    - forward(): prediction uses content classifier only, same as DomainBed SagNet.
    """
    def __init__(
        self,
        model_path: str,
        num_classes: int,
        pretrained: bool = False,
        sag_w_adv: float = 0.1,
    ):
        super().__init__()

        self.network_f = TimmFeaturizer(
            model_path=model_path,
            pretrained=pretrained,
        )

        self.network_c = nn.Linear(self.network_f.n_outputs, num_classes)
        self.network_s = nn.Linear(self.network_f.n_outputs, num_classes)

        self.weight_adv = sag_w_adv

    def randomize(self, x, what="style", eps=1e-5):
        """
        DomainBed/SagNet feature randomization.

        what="style": keep normalized content, swap/mix feature statistics.
        what="content": keep feature statistics, swap normalized content.
        """
        sizes = x.size()
        alpha = torch.rand(
            sizes[0],
            1,
            device=x.device,
            dtype=x.dtype,
        )

        if len(sizes) == 4:
            x = x.view(sizes[0], sizes[1], -1)
            alpha = alpha.unsqueeze(-1)

        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True)  # match DomainBed-v2 SagNet default

        x = (x - mean) / (var + eps).sqrt()

        idx_swap = torch.randperm(sizes[0], device=x.device)

        if what == "style":
            mean = alpha * mean + (1.0 - alpha) * mean[idx_swap]
            var = alpha * var + (1.0 - alpha) * var[idx_swap]
        elif what == "content":
            x = x[idx_swap].detach()
        else:
            raise ValueError("what must be either 'style' or 'content'.")

        x = x * (var + eps).sqrt() + mean

        return x.view(*sizes)

    def forward_c(self, x):
        # Learn content classifier on randomized style.
        features = self.network_f(x)
        features = self.randomize(features, what="style")
        return self.network_c(features)

    def forward_s(self, x):
        # Learn style classifier on randomized content.
        features = self.network_f(x)
        features = self.randomize(features, what="content")
        return self.network_s(features)

    def predict(self, x):
        return self.network_c(self.network_f(x))

    def forward(self, x, mode="predict"):
        # DataParallel can call this with mode="content"/"style".
        if mode == "content":
            return self.forward_c(x)
        if mode == "style":
            return self.forward_s(x)
        if mode == "predict":
            return self.predict(x)
        raise ValueError("mode must be one of: 'predict', 'content', 'style'.")


def create_model(model_path: str, num_classes: int, pretrained: bool = False):
    """
    Create a normal ERM classification model.

    - pretrained=False: train the whole model from scratch.
    - Keep pretrained=False for the paper-comparison from-scratch MobileNetV4 run.
    - All trainable parameters are optimized.
    """
    if model_path is None:
        raise ValueError("model_path must be specified.")
    if num_classes is None:
        raise ValueError("num_classes must be specified.")

    model = timm.create_model(
        model_path,
        pretrained=pretrained,
        num_classes=num_classes,
    )

    # Robust classifier reset for timm backbones.
    if hasattr(model, "reset_classifier"):
        model.reset_classifier(num_classes=num_classes)
    elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Linear(
            in_features=model.classifier.in_features,
            out_features=num_classes,
            bias=True,
        )
    elif hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(
            in_features=model.fc.in_features,
            out_features=num_classes,
            bias=True,
        )
    else:
        raise ValueError(
            "Could not find/reset classifier head. "
            "Check the timm model head name manually."
        )

    return model


def create_sagnet_model(
    model_path: str,
    num_classes: int,
    pretrained: bool = False,
    sag_w_adv: float = 0.1,
):
    """
    Create a normal SagNet model.

    This does not add MoCo or any contrastive component. If pretrained=False,
    the whole SagNet model is trained from scratch.
    """
    if model_path is None:
        raise ValueError("model_path must be specified.")
    if num_classes is None:
        raise ValueError("num_classes must be specified.")

    return SagNetModel(
        model_path=model_path,
        num_classes=num_classes,
        pretrained=pretrained,
        sag_w_adv=sag_w_adv,
    )


In [ ]:
class Trainer:
    def __init__(
        self,
        model_path: str = None,
        num_classes: int = 60,
        pretrained: bool = False,
        seed: int = 42,
        root_path: str = "/kaggle/input/datasets/trieuvuongnguyen/nicopp-rethink-splitted/NICO++",
        target_domain=None,
        source_transform=None,
        target_transform=None,
        batch_size: int = 32,
        lr: float = 1e-4,
        weight_decay: float = 1e-4,
        max_iters: int = 10000,
        val_ratio: float = 0.2,
        source_splits=("train",),
        target_splits=("test",),
        num_workers: int = 2,
        use_amp: bool = True,
        use_data_parallel: bool = True,
        algorithm: str = "ERM",
        sag_w_adv: float = 0.1,
    ):
        """
        DomainBed/DomainBed-v2-style trainer for ERM or normal SagNet.

        Important settings:
        - One minibatch is sampled from EACH source domain.
        - ERM concatenates all source minibatches and applies cross-entropy.
        - SagNet uses content loss, style loss, and adversarial style confusion.
        - Model selection uses only held-out validation data from SOURCE domains.
        - Target domains are evaluated separately and are not used to select checkpoints.
        - For NICO++ trained from scratch, the paper appendix uses 60,000 iterations.
        """

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.seed = seed
        set_seed(self.seed)

        self.root_path = root_path
        self.target_domain = target_domain

        self.model_path = model_path
        self.num_classes = num_classes
        self.pretrained = pretrained

        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.max_iters = max_iters

        algorithm = algorithm.lower()
        if algorithm not in ["erm", "sagnet"]:
            raise ValueError("algorithm must be either 'ERM' or 'SagNet'.")

        self.algorithm = "SagNet" if algorithm == "sagnet" else "ERM"
        self.sag_w_adv = sag_w_adv

        self.val_ratio = val_ratio
        self.source_splits = tuple(source_splits)
        self.target_splits = tuple(target_splits)

        self.num_workers = num_workers
        self.use_amp = use_amp and torch.cuda.is_available()
        self.use_data_parallel = use_data_parallel

        self.scaler = GradScaler("cuda", enabled=self.use_amp)

        self.best_val_acc = -1.0
        self.best_state_dict = None
        self.history = []

        all_domain_groups = sorted([
            d.name for d in Path(root_path).iterdir()
            if d.is_dir()
        ])

        if target_domain is None:
            raise ValueError("target_domain must be specified.")

        if target_domain not in all_domain_groups:
            raise ValueError(
                f"target_domain '{target_domain}' not found. "
                f"Available domain groups: {all_domain_groups}"
            )

        self.source_domain_groups = [
            d for d in all_domain_groups
            if d != target_domain
        ]

        print(f"Target group: {self.target_domain}")
        print(f"Source groups: {self.source_domain_groups}")
        print(f"Source splits used for train/val: {self.source_splits}")
        print(f"Target splits used for final eval: {self.target_splits}")

        # Build one global class mapping from class-folder names.
        # This does not load target images into training.
        class_index_dataset = NICOPPDataset(
            root_path=self.root_path,
            domain_groups=all_domain_groups,
            splits=("train", "test"),
            transform=None,
        )
        self.class_to_idx = class_index_dataset.class_to_idx

        if len(self.class_to_idx) != self.num_classes:
            print(
                f"Warning: num_classes={self.num_classes}, "
                f"but found {len(self.class_to_idx)} class folders. "
                f"Using found value."
            )
            self.num_classes = len(self.class_to_idx)

        print(f"Number of classes: {self.num_classes}")

        # Build model.
        if self.algorithm == "SagNet":
            model = create_sagnet_model(
                model_path=self.model_path,
                num_classes=self.num_classes,
                pretrained=self.pretrained,
                sag_w_adv=self.sag_w_adv,
            ).to(self.device)
        else:
            model = create_model(
                model_path=self.model_path,
                num_classes=self.num_classes,
                pretrained=self.pretrained,
            ).to(self.device)

        if self.use_data_parallel and torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
            model = nn.DataParallel(model)
        else:
            print(f"Using device: {self.device}")

        self.model = model

        # Build one train loader per real source domain.
        # This mirrors DomainBed-v2: minibatches remain domain-separated until ERM concatenates them.
        self.source_train_loaders = []
        self.source_val_subsets = []
        self.source_domain_names = []

        domain_counter = 0

        for domain_group in self.source_domain_groups:
            real_domains = self._get_real_domains(domain_group, self.source_splits)

            for real_domain in real_domains:
                train_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=self.source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=source_transform,
                )

                val_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=self.source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=target_transform,
                )

                n = len(train_dataset_full)

                if n == 0:
                    raise ValueError(
                        f"Empty source domain: group={domain_group}, real_domain={real_domain}"
                    )

                val_size = int(n * self.val_ratio)
                val_size = max(1, val_size)
                train_size = n - val_size

                if train_size <= 0:
                    raise ValueError(
                        f"Not enough samples for source domain: "
                        f"group={domain_group}, real_domain={real_domain}, n={n}"
                    )

                generator = torch.Generator().manual_seed(self.seed + domain_counter)
                indices = torch.randperm(n, generator=generator).tolist()

                train_indices = indices[:train_size]
                val_indices = indices[train_size:]

                train_subset = Subset(train_dataset_full, train_indices)
                val_subset = Subset(val_dataset_full, val_indices)

                # DomainBed-v2 uses one per-domain batch size.
                # If a tiny domain has fewer samples than batch_size, do not drop its only batch.
                drop_last = len(train_subset) >= self.batch_size

                loader_kwargs = dict(
                    batch_size=self.batch_size,
                    shuffle=True,
                    num_workers=self.num_workers,
                    pin_memory=torch.cuda.is_available(),
                    drop_last=drop_last,
                )

                if self.num_workers > 0:
                    loader_kwargs.update(
                        persistent_workers=True,
                        prefetch_factor=2,
                    )

                train_loader = DataLoader(train_subset, **loader_kwargs)

                if len(train_loader) == 0:
                    raise ValueError(
                        f"Zero batches for source domain {domain_group}/{real_domain}. "
                        f"Reduce batch_size."
                    )

                self.source_train_loaders.append(train_loader)
                self.source_val_subsets.append(val_subset)
                self.source_domain_names.append(f"{domain_group}/{real_domain}")

                print(
                    f"Source real domain: {domain_group}/{real_domain:8s} "
                    f"| train={len(train_subset)} "
                    f"| val={len(val_subset)} "
                    f"| batches={len(train_loader)}"
                )

                domain_counter += 1

        if len(self.source_train_loaders) == 0:
            raise ValueError("No source loaders were created.")

        self.source_val_dataset = ConcatDataset(self.source_val_subsets)

        if len(self.source_val_dataset) == 0:
            raise ValueError(
                "Source validation dataset is empty. "
                "Increase val_ratio or check your source datasets."
            )

        eval_batch_size = self.batch_size * len(self.source_train_loaders)

        self.source_val_loader = DataLoader(
            self.source_val_dataset,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

        print(f"Source val samples: {len(self.source_val_dataset)}")

        # Target test loaders: one loader per real target domain.
        self.target_real_domains = self._get_real_domains(target_domain, self.target_splits)

        if len(self.target_real_domains) == 0:
            raise ValueError(
                f"No real target domains found in target group={target_domain}. "
                f"Check folder structure and target_splits={self.target_splits}."
            )

        self.target_datasets = {}
        self.target_loaders = {}

        for real_domain in self.target_real_domains:
            target_dataset = NICOPPDataset(
                root_path=self.root_path,
                domain_groups=[target_domain],
                splits=self.target_splits,
                real_domains=[real_domain],
                class_to_idx=self.class_to_idx,
                transform=target_transform,
            )

            if len(target_dataset) == 0:
                raise ValueError(
                    f"Empty target domain: group={target_domain}, real_domain={real_domain}"
                )

            target_loader = DataLoader(
                target_dataset,
                batch_size=eval_batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=torch.cuda.is_available(),
            )

            self.target_datasets[real_domain] = target_dataset
            self.target_loaders[real_domain] = target_loader

            print(
                f"Target real domain: {real_domain:8s} "
                f"| samples={len(target_dataset)}"
            )

        self.target_dataset = ConcatDataset(list(self.target_datasets.values()))

        self.target_loader = DataLoader(
            self.target_dataset,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

        print(f"Target group samples: {len(self.target_dataset)}")
        print(f"Batch size per source domain: {self.batch_size}")
        print(f"Effective train batch size: {self.batch_size * len(self.source_train_loaders)}")

        self.criterion = nn.CrossEntropyLoss()

        if self.algorithm == "SagNet":
            base_model = self.get_base_model()

            self.optimizer_f = Adam(
                base_model.network_f.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )
            self.optimizer_c = Adam(
                base_model.network_c.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )
            self.optimizer_s = Adam(
                base_model.network_s.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )

            self.optimizers = [
                self.optimizer_f,
                self.optimizer_c,
                self.optimizer_s,
            ]
        else:
            self.optimizer = Adam(
                self.model.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )

            self.optimizers = [self.optimizer]

        # DomainBed-v2 computes steps_per_epoch from the smallest source domain.
        # Use ceil(dataset_size / batch_size), not len(loader), to match its train.py logic.
        self.steps_per_epoch = math.ceil(
            min(len(loader.dataset) for loader in self.source_train_loaders) / self.batch_size
        )

        if self.steps_per_epoch <= 0:
            raise ValueError("At least one source train loader has zero batches. Try reducing batch_size.")

        self.max_epochs = math.ceil(self.max_iters / self.steps_per_epoch)

        self.schedulers = [
            CosineAnnealingLR(
                optimizer,
                T_max=self.max_epochs,
            )
            for optimizer in self.optimizers
        ]

        # Backward-compatible alias used by old ERM-only code.
        self.scheduler = self.schedulers[0]

        print(f"Algorithm: {self.algorithm}")
        if self.algorithm == "SagNet":
            print(f"SagNet adversarial weight sag_w_adv: {self.sag_w_adv}")
        print(f"Steps per epoch: {self.steps_per_epoch}")
        print(f"Max epochs for scheduler: {self.max_epochs}")

    def _get_real_domains(self, domain_group, splits):
        real_domains = set()

        for split in splits:
            split_path = Path(self.root_path) / domain_group / split
            if not split_path.exists():
                continue

            for real_domain_dir in split_path.iterdir():
                if real_domain_dir.is_dir():
                    real_domains.add(real_domain_dir.name)

        return sorted(real_domains)

    def get_base_model(self):
        if isinstance(self.model, nn.DataParallel):
            return self.model.module
        return self.model

    def _current_lr(self):
        return self.optimizers[0].param_groups[0]["lr"]

    def _set_train_mode(self):
        # Keep the whole model in train mode for ERM/SagNet.
        self.model.train()

    def evaluate(self, loader):
        self.model.eval()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)

                with autocast("cuda", enabled=self.use_amp):
                    logits = self.model(images)
                    loss = self.criterion(logits, labels)

                preds = logits.argmax(dim=1)

                cur_batch_size = labels.size(0)
                total_loss += loss.item() * cur_batch_size
                total_correct += (preds == labels).sum().item()
                total_samples += cur_batch_size

        avg_loss = total_loss / total_samples
        avg_acc = total_correct / total_samples

        return {
            "loss": avg_loss,
            "acc": avg_acc,
            "total_correct": total_correct,
            "total": total_samples,
        }

    def _next_source_minibatches(self, source_iters):
        minibatches = []

        for i, loader in enumerate(self.source_train_loaders):
            try:
                images, labels = next(source_iters[i])
            except StopIteration:
                source_iters[i] = iter(loader)
                images, labels = next(source_iters[i])

            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            minibatches.append((images, labels))

        return minibatches

    def _erm_update(self, minibatches):
        # Same ERM logic as DomainBed-v2:
        # all_x = cat([x for x, y in minibatches])
        # all_y = cat([y for x, y in minibatches])
        all_x = torch.cat([x for x, y in minibatches], dim=0)
        all_y = torch.cat([y for x, y in minibatches], dim=0)

        with autocast("cuda", enabled=self.use_amp):
            logits = self.model(all_x)
            loss = self.criterion(logits, all_y)

        self.optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.step(self.optimizer)
        self.scaler.update()

        preds = logits.argmax(dim=1)
        train_acc = (preds == all_y).float().mean().item()

        return loss.item(), train_acc, all_y.size(0), {
            "loss": loss.item(),
        }

    def _sagnet_update(self, minibatches):
        """
        DomainBed-style normal SagNet update.

        Algorithm steps:
        1. Learn content classifier + featurizer on style-randomized features.
        2. Learn style classifier on content-randomized features.
        3. Learn featurizer adversarially so style predictions become uncertain.

        No MoCo or contrastive objective is added.
        """
        all_x = torch.cat([x for x, y in minibatches], dim=0)
        all_y = torch.cat([y for x, y in minibatches], dim=0)

        # 1) Learn content.
        self.optimizer_f.zero_grad(set_to_none=True)
        self.optimizer_c.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=self.use_amp):
            logits_c = self.model(all_x, mode="content")
            loss_c = self.criterion(logits_c, all_y)

        self.scaler.scale(loss_c).backward()
        self.scaler.step(self.optimizer_f)
        self.scaler.step(self.optimizer_c)
        self.scaler.update()

        # 2) Learn style.
        self.optimizer_s.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=self.use_amp):
            logits_s = self.model(all_x, mode="style")
            loss_s = self.criterion(logits_s, all_y)

        self.scaler.scale(loss_s).backward()
        self.scaler.step(self.optimizer_s)
        self.scaler.update()

        # 3) Learn adversary: update only the featurizer.
        self.optimizer_f.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=self.use_amp):
            logits_adv = self.model(all_x, mode="style")
            loss_adv = -torch.nn.functional.log_softmax(
                logits_adv,
                dim=1,
            ).mean(1).mean()
            loss_adv = loss_adv * self.sag_w_adv

        self.scaler.scale(loss_adv).backward()
        self.scaler.step(self.optimizer_f)
        self.scaler.update()

        preds = logits_c.argmax(dim=1)
        train_acc = (preds == all_y).float().mean().item()

        total_loss = loss_c.detach() + loss_s.detach() + loss_adv.detach()

        return total_loss.item(), train_acc, all_y.size(0), {
            "loss": total_loss.item(),
            "loss_c": loss_c.item(),
            "loss_s": loss_s.item(),
            "loss_adv": loss_adv.item(),
        }

    def _update(self, minibatches):
        if self.algorithm == "SagNet":
            return self._sagnet_update(minibatches)
        return self._erm_update(minibatches)


    def train(
        self,
        eval_iters=600,
        log_iters=20,
        save_path=None,
        eval_target_during_training=False,
    ):
        self._set_train_mode()

        print(f"Training {self.algorithm} with {self.max_iters} iterations")
        print(f"Target group: {self.target_domain}")
        print(f"Source real domains: {self.source_domain_names}")
        print(f"Validation/checkpoint frequency: every {eval_iters} iterations")
        print(f"Logging frequency: every {log_iters} iterations")

        source_iters = [iter(loader) for loader in self.source_train_loaders]

        best_val_acc = -1.0
        best_state_dict = None
        history = []

        running_loss = 0.0
        running_acc_sum = 0.0
        running_steps = 0

        pbar = tqdm(range(1, self.max_iters + 1), total=self.max_iters)

        for step in pbar:
            minibatches = self._next_source_minibatches(source_iters)
            loss_value, train_acc, seen_samples, update_metrics = self._update(minibatches)

            running_loss += loss_value
            running_acc_sum += train_acc
            running_steps += 1

            # DomainBed-v2 steps the scheduler once per epoch.
            if step % self.steps_per_epoch == 0:
                for scheduler in self.schedulers:
                    scheduler.step()

            if step % log_iters == 0:
                train_loss = running_loss / running_steps
                train_acc_avg = running_acc_sum / running_steps

                pbar.set_postfix({
                    "loss": f"{train_loss:.4f}",
                    "train_acc": f"{train_acc_avg:.4f}",
                    "lr": f"{self._current_lr():.6f}",
                })

                running_loss = 0.0
                running_acc_sum = 0.0
                running_steps = 0

            if step % eval_iters == 0 or step == self.max_iters:
                source_val_metrics = self.evaluate(self.source_val_loader)

                record = {
                    "iter": step,
                    "source_val_loss": source_val_metrics["loss"],
                    "source_val_acc": source_val_metrics["acc"],
                    "lr": self._current_lr(),
                }

                print(
                    f"\nIter [{step}/{self.max_iters}] "
                    f"| Source Val Loss: {source_val_metrics['loss']:.4f} "
                    f"| Source Val Acc: {source_val_metrics['acc']:.4f} "
                    f"| LR: {self._current_lr():.6f}"
                )

                # DomainBed-v2 logs test performance at checkpoints, but DOES NOT
                # use it for model selection. Best model is selected only by source val.
                if eval_target_during_training:
                    target_metrics = self.evaluate_target(load_best=False, verbose=False)
                    record["target_weighted_acc"] = target_metrics["weighted_acc"]
                    record["target_macro_acc"] = target_metrics["macro_acc"]
                    record["target_domain_acc"] = {
                        k: v["acc"] for k, v in target_metrics["domain_metrics"].items()
                    }
                    print(
                        f"Target Weighted Acc: {target_metrics['weighted_acc']:.4f} "
                        f"| Target Macro Acc: {target_metrics['macro_acc']:.4f}"
                    )

                history.append(record)

                if source_val_metrics["acc"] > best_val_acc:
                    best_val_acc = source_val_metrics["acc"]
                    best_state_dict = {
                        k: v.detach().cpu().clone()
                        for k, v in self.get_base_model().state_dict().items()
                    }

                    print(f"New best source-val acc at iter {step}: {best_val_acc:.4f}")

                    if save_path is not None:
                        torch.save({
                            "iter": step,
                            "model_state_dict": best_state_dict,
                            "best_val_acc": best_val_acc,
                            "class_to_idx": self.class_to_idx,
                            "source_domain_groups": self.source_domain_groups,
                            "source_domain_names": self.source_domain_names,
                            "target_domain": self.target_domain,
                            "target_real_domains": self.target_real_domains,
                            "source_splits": self.source_splits,
                            "target_splits": self.target_splits,
                            "model_path": self.model_path,
                            "num_classes": self.num_classes,
                            "pretrained": self.pretrained,
                            "algorithm": self.algorithm,
                            "sag_w_adv": self.sag_w_adv,
                            "lr": self.lr,
                            "weight_decay": self.weight_decay,
                            "batch_size_per_domain": self.batch_size,
                            "max_iters": self.max_iters,
                            "eval_iters": eval_iters,
                            "seed": self.seed,
                        }, save_path)

                self._set_train_mode()

        self.best_val_acc = best_val_acc
        self.best_state_dict = best_state_dict
        self.history = history

        print(f"\nBest Source Val Acc: {best_val_acc:.4f}")

        return history

    def evaluate_target(self, load_best=True, verbose=True):
        if load_best and self.best_state_dict is not None:
            self.get_base_model().load_state_dict(self.best_state_dict)

        self.model.eval()

        domain_metrics = {}

        for real_domain, loader in self.target_loaders.items():
            metrics = self.evaluate(loader)
            domain_metrics[real_domain] = metrics

        # Macro average across target domains: useful for paper-style domain columns.
        macro_acc = sum(m["acc"] for m in domain_metrics.values()) / len(domain_metrics)
        macro_loss = sum(m["loss"] for m in domain_metrics.values()) / len(domain_metrics)

        # Weighted average across target samples: this matches DomainBed-v2 "overall".
        total_correct = sum(m["total_correct"] for m in domain_metrics.values())
        total_samples = sum(m["total"] for m in domain_metrics.values())

        weighted_acc = total_correct / total_samples
        weighted_loss = sum(
            m["loss"] * m["total"] for m in domain_metrics.values()
        ) / total_samples

        if verbose:
            print("\nTarget Evaluation")
            print(f"Target group: {self.target_domain}")

            for real_domain, metrics in domain_metrics.items():
                print(
                    f"{real_domain:8s} "
                    f"| Loss: {metrics['loss']:.4f} "
                    f"| Acc: {metrics['acc']:.4f} "
                    f"| Correct: {metrics['total_correct']} / {metrics['total']}"
                )

            print(f"Macro Acc across target domains: {macro_acc:.4f}")
            print(f"Weighted Acc across target samples / DomainBed-v2 overall: {weighted_acc:.4f}")

        return {
            "target_group": self.target_domain,
            "domain_metrics": domain_metrics,
            "macro_loss": macro_loss,
            "macro_acc": macro_acc,
            "weighted_loss": weighted_loss,
            "weighted_acc": weighted_acc,
            "total_correct": total_correct,
            "total": total_samples,
        }


In [ ]:
source_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
    transforms.RandomGrayscale(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
trainer = Trainer(
    model_path="mobilenetv4_conv_small.e2400_r224_in1k",
    num_classes=60,
    pretrained=False,                  # normal SagNet from scratch; no supervised/ImageNet or MoCo init
    algorithm="SagNet",                 # "ERM" or "SagNet"
    sag_w_adv=0.1,                      # DomainBed default for SagNet
    target_domain="autumn_rock",        # NICO++ leave-one-group-out: autumn + rock are target
    source_transform=source_transform,
    target_transform=target_transform,
    batch_size=32,                      # per-source-domain batch size
    lr=1e-4,
    weight_decay=1e-4,
    max_iters=10000,                    # DomainBed-v2 NICO++ setting from the paper
    val_ratio=0.2,                      # source-domain validation holdout
    source_splits=("train",),           # source train is internally split into train/val
    target_splits=("test",),            # target is held out for final evaluation
    num_workers=2,
    use_amp=True,
    use_data_parallel=True,             # use both Kaggle T4 GPUs when available
)


In [ ]:
history = trainer.train(
    eval_iters=600,
    log_iters=100,
    save_path="/kaggle/working/best_sagnet.pth",
    eval_target_during_training=False,  # keep target test unseen during training/checkpoint selection
)


In [ ]:
NICO_PP_TARGET_GROUPS = ["autumn_rock", "dim_grass", "outdoor_water"]
NICO_PP_PAPER_DOMAIN_ORDER = ["autumn", "rock", "dim", "grass", "outdoor", "water"]


def paper_metric_row(results_by_group, algorithm_name=None):
    """
    Return NICO++ paper-style columns: each real target-domain accuracy in percent
    plus the macro Avg across available target domains.

    Pass either one evaluate_target() result or a dict such as:
        {"autumn_rock": autumn_rock_results, "dim_grass": dim_grass_results, ...}
    """
    if isinstance(results_by_group, dict) and (
        "domain_metrics" in results_by_group or "per_domain" in results_by_group
    ):
        grouped_results = {results_by_group.get("target_group", "target"): results_by_group}
    else:
        grouped_results = results_by_group

    accs = {}
    for _, result in grouped_results.items():
        if isinstance(result, dict) and "target_results" in result:
            result = result["target_results"]

        domain_metrics = result.get("domain_metrics", result.get("per_domain", {}))
        for domain, metrics in domain_metrics.items():
            real_domain = str(domain).split("/")[-1]
            acc = float(metrics["acc"])
            accs[real_domain] = acc * 100.0 if acc <= 1.0 else acc

    row = {domain: accs.get(domain, np.nan) for domain in NICO_PP_PAPER_DOMAIN_ORDER}
    values = list(row.values())
    row["Avg"] = float(np.nanmean(values)) if np.isfinite(values).any() else np.nan

    if algorithm_name is not None:
        row = {"Algorithm": algorithm_name, **row}

    return pd.DataFrame([row])


In [ ]:
ckpt_path = "/kaggle/working/best_sagnet.pth"

checkpoint = torch.load(ckpt_path, map_location="cpu")
state_dict = checkpoint["model_state_dict"]

trainer.get_base_model().load_state_dict(state_dict)

target_results = trainer.evaluate_target()

paper_metrics = paper_metric_row(target_results, algorithm_name="SagNet")
paper_metrics
